In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tf2onnx
import onnx

2024-11-18 09:50:04.780624: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-18 09:50:04.805937: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731941404.822323 2048518 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731941404.827389 2048518 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-18 09:50:04.845650: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
inshp = (21, 4, 9)
units = 45
a = keras.layers.Input(shape=inshp)
b = keras.layers.Dense(units, activation='relu')(a)
model = keras.models.Model(inputs=a, outputs=b)


2024-11-18 09:50:32.445530: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:
input_signature = [tf.TensorSpec(shape=model.input.shape, dtype=tf.float32)]
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature)

I0000 00:00:1731941445.644442 2048518 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1731941445.644587 2048518 single_machine.cc:361] Starting new session
I0000 00:00:1731941445.693756 2048518 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1731941445.693874 2048518 single_machine.cc:361] Starting new session
rewriter <function rewrite_constant_fold at 0x14eb488b3ce0>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.


In [20]:
print(model.input)
print(model.layers[0].input)
print(model.layers[0].output)
print(model.layers[1].input)
print(model.layers[1].output)
print(model.output)

<KerasTensor shape=(None, 21, 4, 9), dtype=float32, sparse=False, name=keras_tensor>
[]
<KerasTensor shape=(None, 21, 4, 9), dtype=float32, sparse=False, name=keras_tensor>
<KerasTensor shape=(None, 21, 4, 9), dtype=float32, sparse=False, name=keras_tensor>
<KerasTensor shape=(None, 21, 4, 45), dtype=float32, sparse=False, name=keras_tensor_1>
<KerasTensor shape=(None, 21, 4, 45), dtype=float32, sparse=False, name=keras_tensor_1>


In [29]:
print(model.layers[1].__class__.__name__)
print(model.layers[1].activation.__name__)

Dense
relu


In [10]:
graph = onnx_model.graph

In [38]:
print(onnx_model)

ir_version: 8
producer_name: "tf2onnx"
producer_version: "1.16.1 15c810"
graph {
  node {
    input: "args_0"
    input: "functional_1/dense_1/Cast/ReadVariableOp:0"
    output: "functional_1/dense_1/MatMul:0"
    name: "functional_1/dense_1/MatMul"
    op_type: "MatMul"
  }
  node {
    input: "functional_1/dense_1/MatMul:0"
    input: "functional_1/dense_1/Add/ReadVariableOp:0"
    output: "functional_1/dense_1/Add:0"
    name: "functional_1/dense_1/Add"
    op_type: "Add"
  }
  node {
    input: "functional_1/dense_1/Add:0"
    output: "dense"
    name: "functional_1/dense_1/Relu"
    op_type: "Relu"
  }
  name: "tf2onnx"
  initializer {
    dims: 9
    dims: 45
    data_type: 1
    name: "functional_1/dense_1/Cast/ReadVariableOp:0"
    raw_data: "\252\352\016>@c\014=\226f\036>L\003\323=\266\224\363\275\000-X\276\350\363\006\275`\030)<\3202\336\274\300\"\037=\206\t!>\331_\212>\341G\214>\366\261\263\2753\334\201\276\3008\342\2750e\242\274\346\000\005>\324.\312=\273\336\251>6\303d\276

In [37]:
nodes = 0
for node in graph.node:
    print(node)
    nodes += 1

input: "args_0"
input: "functional_1/dense_1/Cast/ReadVariableOp:0"
output: "functional_1/dense_1/MatMul:0"
name: "functional_1/dense_1/MatMul"
op_type: "MatMul"

input: "functional_1/dense_1/MatMul:0"
input: "functional_1/dense_1/Add/ReadVariableOp:0"
output: "functional_1/dense_1/Add:0"
name: "functional_1/dense_1/Add"
op_type: "Add"

input: "functional_1/dense_1/Add:0"
output: "dense"
name: "functional_1/dense_1/Relu"
op_type: "Relu"



In [33]:
print(graph.activation_function)

AttributeError: 'GraphProto' object has no attribute 'activation_function'

In [15]:
for node in graph.node:
    # node inputs
    for idx, node_input_name in enumerate(node.input):
        print('input: ', idx, node_input_name)
    # node outputs
    for idx, node_output_name in enumerate(node.output):
        print('output:', idx, node_output_name)

input:  0 args_0
input:  1 functional_1/dense_1/Cast/ReadVariableOp:0
output: 0 functional_1/dense_1/MatMul:0
input:  0 functional_1/dense_1/MatMul:0
input:  1 functional_1/dense_1/Add/ReadVariableOp:0
output: 0 functional_1/dense_1/Add:0
input:  0 functional_1/dense_1/Add:0
output: 0 dense


In [17]:
INTIALIZERS  = onnx_model.graph.initializer
onnx_weights = {}
for initializer in INTIALIZERS:
    W = onnx.numpy_helper.to_array(initializer)
    onnx_weights[initializer.name] = W

In [18]:
print(onnx_weights.keys())

dict_keys(['functional_1/dense_1/Cast/ReadVariableOp:0', 'functional_1/dense_1/Add/ReadVariableOp:0'])
